# Circuit clusters — T5 degradation transfer

Circuits the 2026 season has not visited yet borrow tyre coefficients from circuits
it has. The clustering runs on era-2024 behaviour every circuit has data for:
degradation amplitude (abrasiveness), full-throttle fraction and braking energy
(layout), and pit-lane loss (a stable circuit fingerprint). k is chosen by
silhouette over the configured range, seeded so a rerun of `make fit` cannot
reshuffle transfer donors. A transferred coefficient carries `low_confidence: true`
and its donors, forever.

In [ ]:
import numpy as np
import pandas as pd
from senna import config
from senna.data.db import database_url, engine_for
from senna.physics.fits import frames, plots
from senna.physics.fits.gate import assert_quality_reports

fits_cfg = config.load("fits")
ingest_cfg = config.load("ingest")
CALIB = config.get(config.load("data_split"), "calibration_rounds")
SLICKS = ("SOFT", "MEDIUM", "HARD")

assert_quality_reports(
    config.get(ingest_cfg, "seasons"),
    config.REPO_ROOT / config.get(ingest_cfg, "quality.report_dir"),
)
engine = engine_for(database_url())
conn = await engine.connect()
index = await frames.circuit_index(conn, CALIB)
index[["slug", "era"]].drop_duplicates().groupby("era").size()

In [ ]:
import yaml

params = yaml.safe_load(open(config.REPO_ROOT / "configs" / "physics_params.yaml"))
feature_rows = {}
for slug, circuit in params["circuits"].items():
    deg_2024 = (circuit.get("degradation") or {}).get(2024) or {}
    energy_2024 = (circuit.get("energy") or {}).get(2024) or {}
    pit = circuit.get("pit_loss") or {}
    if not (deg_2024 and energy_2024 and pit):
        continue
    feature_rows[slug] = {
        "deg_a_2024": float(np.mean([row["a"] for row in deg_2024.values()])),
        "full_throttle_frac": energy_2024["full_throttle_frac"],
        "braking_energy_mj_per_lap": energy_2024["braking_energy_mj_per_lap"],
        "pit_loss_s": pit["mean_s"],
    }
features = pd.DataFrame.from_dict(feature_rows, orient="index")
features

In [ ]:
from senna.physics.fits.clusters import assign_clusters

clustering = assign_clusters(
    features,
    k_range=tuple(config.get(fits_cfg, "clusters.k_range")),
    seed=config.get(fits_cfg, "clusters.seed"),
)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(list(clustering.silhouettes), list(clustering.silhouettes.values()), marker="o")
ax.axvline(clustering.k, color="crimson", linewidth=0.8)
ax.set_xlabel("k")
ax.set_ylabel("silhouette")
print(plots.save_figure(fig, "clusters", "silhouette_curve"))

assignments = pd.Series(clustering.assignments, name="cluster").sort_values()
stored = pd.Series(
    {s: v for s, v in params["clusters"]["assignments"].items()}, name="stored"
)
pd.concat([assignments, stored], axis=1)

The stored column confirms the yaml carries exactly this clustering.

In [ ]:
transfer_rows = []
for slug, circuit in params["circuits"].items():
    for compound, row in ((circuit.get("degradation") or {}).get(2026) or {}).items():
        if isinstance(row, dict) and row.get("source_circuit"):
            transfer_rows.append(
                {
                    "circuit": slug,
                    "compound": compound,
                    "a": row["a"],
                    "b": row["b"],
                    "donors": row["source_circuit"],
                }
            )
pd.DataFrame(transfer_rows)

In [ ]:
await conn.close()
await engine.dispose()